# NeoJaundice model audit and training evidence

This notebook verifies recorded local evidence. Labels came from dataset folder names and are not clinically verified. Similarity groups are not patient IDs. No patient images are embedded.


In [ ]:
from pathlib import Path
import csv, json
from collections import Counter
PROJECT = Path.cwd()
if PROJECT.name == 'ml': PROJECT = PROJECT.parent
assert (PROJECT / 'models/research/evaluation.json').exists(), 'Run from the project root or ml folder'


In [ ]:
audit = json.loads((PROJECT / 'ml/evidence/audit.json').read_text())
print({key: audit[key] for key in ['total_images', 'class_counts', 'exact_duplicate_extra_images', 'similarity_groups', 'excluded_images']})
print('Unreadable images:', len(audit['invalid_images']))


In [ ]:
rows = list(csv.DictReader((PROJECT / 'ml/evidence/split_assignments.csv').open()))
groups = {part: {row['group_id'] for row in rows if row['partition'] == part} for part in ['train', 'validation', 'test']}
assert not groups['train'] & groups['validation']
assert not groups['train'] & groups['test']
assert not groups['validation'] & groups['test']
print('Images by partition:', dict(Counter(row['partition'] for row in rows)))
print('Detected groups do not cross partitions. Same-patient separation remains unverified.')


In [ ]:
evaluation = json.loads((PROJECT / 'models/research/evaluation.json').read_text())
metrics = evaluation['partitions']['test']['image_metrics']
print('Held-out images:', metrics['count'])
print('Accuracy:', metrics['classification_report']['accuracy'])
print('Balanced accuracy:', metrics['balanced_accuracy'])
print('Jaundice-folder recall:', metrics['classification_report']['jaundice']['recall'])
print('Confusion matrix, row truth / column prediction, normal then jaundice:', metrics['confusion_matrix_normal_jaundice'])
print('Limitations:', evaluation['limitations'])


## Reproduction

Run `ml/audit.py` on your permitted dataset copy, then `ml/train.py` as described in ml/README.md. Add a completed patient manifest to obtain a patient-grouped split. This delivery has no prospective, external or clinical validation.
